# Customer Intelligence System
### End-to-End Country Segmentation using EDA, K-Means, DBSCAN, Random Forest and XGBoost

## 1. Import Libraries

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

from xgboost import XGBClassifier

import warnings
warnings.filterwarnings('ignore')


## 2. Load Dataset

In [ ]:

df = pd.read_csv('Country-data.csv')
df.head()


## 3. Dataset Overview

In [ ]:

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns)

print("\nData Types:")
print(df.dtypes)

df.info()


## 4. Missing Values and Duplicate Check

In [ ]:

print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())


## 5. Statistical Summary

In [ ]:

df.describe()


## 6. EDA - Histograms

In [ ]:

num_cols = df.drop('country', axis=1).columns

for col in num_cols:
    plt.figure(figsize=(6,4))
    sns.histplot(df[col], kde=True)
    plt.title(f'Distribution of {col}')
    plt.show()


## 7. Outlier Detection using Boxplots

In [ ]:

for col in num_cols:
    plt.figure(figsize=(6,3))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot of {col}')
    plt.show()


## 8. Correlation Heatmap

In [ ]:

corr = df.drop('country', axis=1).corr()

plt.figure(figsize=(12,8))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Heatmap')
plt.show()


## 9. Scatter Plot Analysis

In [ ]:

plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='income', y='gdpp')
plt.title('Income vs GDP')
plt.show()

plt.figure(figsize=(7,5))
sns.scatterplot(data=df, x='child_mort', y='life_expec')
plt.title('Child Mortality vs Life Expectancy')
plt.show()


## 10. Data Preprocessing

In [ ]:

country_names = df['country']

X = df.drop('country', axis=1)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(X_scaled.shape)


## 11. Elbow Method

In [ ]:

inertia = []

for k in range(2,11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertia.append(km.inertia_)

plt.figure(figsize=(8,5))
plt.plot(range(2,11), inertia, marker='o')
plt.xlabel('Number of Clusters')
plt.ylabel('Inertia')
plt.title('Elbow Method')
plt.show()


## 12. K-Means Clustering

In [ ]:

kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)

df['Cluster'] = kmeans.fit_predict(X_scaled)

print(df['Cluster'].value_counts())
print('Silhouette Score:', silhouette_score(X_scaled, df['Cluster']))


## 13. Cluster Profiling

In [ ]:

cluster_profile = df.groupby('Cluster').mean(numeric_only=True)
cluster_profile


## 14. DBSCAN Clustering

In [ ]:

dbscan = DBSCAN(eps=1.5, min_samples=5)

df['DBSCAN_Cluster'] = dbscan.fit_predict(X_scaled)

print(df['DBSCAN_Cluster'].value_counts())


## 15. PCA Visualization

In [ ]:

pca = PCA(n_components=2)

X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(8,6))
plt.scatter(X_pca[:,0], X_pca[:,1], c=df['Cluster'])
plt.xlabel('PCA 1')
plt.ylabel('PCA 2')
plt.title('KMeans Clusters')
plt.show()


## 16. Random Forest Classification

In [ ]:

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    df['Cluster'],
    test_size=0.2,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

print('Random Forest Accuracy:', accuracy_score(y_test, rf_pred))
print(classification_report(y_test, rf_pred))


## 17. XGBoost Classification

In [ ]:

xgb = XGBClassifier(
    n_estimators=200,
    max_depth=4,
    learning_rate=0.1,
    objective='multi:softmax',
    num_class=3,
    random_state=42
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

print('XGBoost Accuracy:', accuracy_score(y_test, xgb_pred))
print(classification_report(y_test, xgb_pred))


## 18. Feature Importance

In [ ]:

importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf.feature_importances_
})

importance = importance.sort_values(
    by='Importance',
    ascending=False
)

print(importance)

plt.figure(figsize=(10,6))
sns.barplot(data=importance, x='Importance', y='Feature')
plt.title('Feature Importance')
plt.show()


## 19. Save Results

In [ ]:

df.to_csv('Country_Intelligence_Output.csv', index=False)
print('Output Saved Successfully')
